# Train Wasserstein Adversarial Network with Gradient Penalty (WGAN-GP) <BR> on the CelebA Dataset

Version 6

Antonio Esteves @ UMinho, Jul 2024
---
TODO:

* Modify `'OUR_WANDB_PROJECT_ID'`
* Modify `'OUR_WANDB_ENTITY'`
* Modify `LOAD_TRAINED_MODEL`
* Modify `SKIP_TRAIN_MODEL`
* In `../config/wgangp_celeba_128x128_05.yaml` file, modify the hyperparameters `experiment_name`, `dataset_path`, `dataset`.
---

In [ ]:
import os
import numpy                  as     np
import PIL.Image              as     Image
from   pathlib                import Path
from   natsort                import natsorted
import matplotlib.pyplot      as     plt
import wandb
import time
import yaml
import math

import torch
from   torch.autograd         import grad
import torch.nn               as     nn
import torchvision.transforms as     transforms
from   torch.utils.data       import DataLoader, Dataset
from   torchvision.utils      import save_image, make_grid

## Configuration

In [ ]:
LOAD_TRAINED_MODEL        = False
SKIP_TRAIN_MODEL          = False

CONFIG_FILE = '../config/wgangp_celeba_128x128_05.yaml'

with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

In [ ]:
if (config['crop_size'] == 'None'):
    config['crop_size'] = None

if (config['gradient_penalty'] == 'None'):
    config['gradient_penalty'] = None

In [ ]:
print('parameters:')
for key, value in config.items():
    print(f'\t{key}: {value}')

## Initializations and create necessary folders

In [ ]:
# Setup device agnostic code

device       = "cuda" if torch.cuda.is_available() else "cpu"

print(f'Using {device} for computing')

train_dir    = Path(config["dataset_path"])

# Location where we will save here the images generated during WGAN-GP training
RESULTS_PATH = f'results/{config["experiment_name"]}'
os.makedirs(RESULTS_PATH, exist_ok=True)

# Location where the trained models will be saved
MODELS_PATH  = os.path.join(os.getcwd(), 'models')
os.makedirs(MODELS_PATH, exist_ok=True)

## Login into Weights & Bias

In [ ]:
wandb.login()

## Track metadata and hyperparameters with Weights & Bias

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = config

wandb.init(
    project = 'OUR_WANDB_PROJECT_ID',
    entity  = 'OUR_WANDB_ENTITY', 
    config  = config_wandb
)

## Utility functions

In [ ]:
def time_format(seconds: int) -> str:
    if seconds is not None:
        seconds = int(seconds)
        d = seconds // (3600 * 24)
        h = seconds // 3600 % 24
        m = seconds % 3600 // 60
        s = seconds % 3600 % 60
        if d > 0:
            return '{:02d}D {:02d}H {:02d}m {:02d}s'.format(d, h, m, s)
        elif h > 0:
            return '{:02d}H {:02d}m {:02d}s'.format(h, m, s)
        elif m > 0:
            return '{:02d}m {:02d}s'.format(m, s)
        elif s > 0:
            return '{:02d}s'.format(s)
    return '-'

## Generator Model

In [ ]:
class Generator(nn.Module):

    def __init__(
            self,
            latent_dim=100,
            generator_channels=64,
            img_size=64,
            img_channels=3,
            g_min_feature_size=4,
            g_conv_layers=4,
        ):

        super(Generator, self).__init__()

        self.n_stride2 = int(math.log2(img_size / g_min_feature_size))
        self.min_feature_size = g_min_feature_size
        
        check_size = (2 ** self.n_stride2) * g_min_feature_size
        
        assert check_size == img_size, \
            print(f'Image size/min_feature_size must be a power of 2')

        assert self.n_stride2 <= g_conv_layers, \
            print(f'Incompatible values for conv layers, min feature size, and image size')

        def dconv_bn_relu(in_dim, out_dim, stride=2, out_pad=1):
            return nn.Sequential(
                nn.ConvTranspose2d(
                    in_channels    = in_dim,
                    out_channels   = out_dim,
                    kernel_size    = 5,
                    stride         = stride,
                    padding        = 2,
                    output_padding = out_pad,
                    bias           = False,
                ),
                nn.BatchNorm2d(out_dim),
                nn.ReLU())

        self.fc = nn.Sequential(
            nn.Linear(
                latent_dim,
                generator_channels * (2 ** (g_conv_layers - 1)) * g_min_feature_size * g_min_feature_size,
                bias = False,
            ),
            nn.BatchNorm1d(generator_channels * (2 ** (g_conv_layers - 1)) * g_min_feature_size * g_min_feature_size),
            nn.ReLU())

        self.conv_layers = nn.ModuleList()

        ch_in  = generator_channels * (2 ** (g_conv_layers - 1))
        ch_out =  int(ch_in / 2)
        ncl    = 0

        for i in range(g_conv_layers-1):
            if ncl < self.n_stride2:
                stride  = 2
                out_pad = 1
            else:
                stride  = 1
                out_pad = 0
            self.conv_layers.append(dconv_bn_relu(ch_in, ch_out, stride, out_pad))
            ch_in   = int(ch_in/2)
            ch_out  = int(ch_out/2)
            ncl    += 1

        if ncl < self.n_stride2:
            stride  = 2
            out_pad = 1
        else:
            stride  = 1
            out_pad = 0

        self.conv_layers.append(
            nn.ConvTranspose2d(
                in_channels    = generator_channels,
                out_channels   = img_channels,
                kernel_size    = 5,
                stride         = stride,
                padding        = 2,
                output_padding = out_pad,
            ),
        )

        self.conv_layers.append(nn.Tanh())

    def forward(self, x):
        y = self.fc(x)
        y = y.view(y.size(0), -1, self.min_feature_size, self.min_feature_size)

        for layer in self.conv_layers:
            y = layer(y)
        return y

## Critic Model

In [ ]:
class Critic(nn.Module):

    def __init__(
            self,
            critic_channels    = 64,
            img_size           = 64,
            img_channels       = 3,
            c_min_feature_size = 4,
            c_conv_layers      = 4,
        ):
        super(Critic, self).__init__()      

        self.n_stride2 = int(math.log2(img_size / c_min_feature_size))
        
        check_size = (2 ** self.n_stride2) * c_min_feature_size

        assert check_size == img_size, \
            print(f'Image size/min_feature_size must be a power of 2')

        assert self.n_stride2 <= c_conv_layers, \
            print(f'Incompatible values for conv layers, min feature size, and image size')

        def conv_ln_lrelu(in_dim, out_dim, stride=2, pad=2):
            return nn.Sequential(
                nn.Conv2d(
                    in_channels  = in_dim,
                    out_channels = out_dim,
                    kernel_size  = 5,
                    stride       = stride,
                    padding      = pad,
                ),
                # Since there is no effective implementation of LayerNorm,
                # we use InstanceNorm2d instead of LayerNorm here.
                nn.InstanceNorm2d(out_dim, affine=True),
                nn.LeakyReLU(0.2)
            )

        self.conv_layers = nn.ModuleList()

        self.conv_layers.append(
            nn.Conv2d(
                in_channels  = img_channels,
                out_channels = critic_channels,
                kernel_size  = 5,
                stride       = 2,
                padding      = 2,
            )
        )
        self.conv_layers.append(nn.LeakyReLU(0.2))

        ch_in  = critic_channels
        ch_out = ch_in * 2
        ncl    = 1

        for i in range(c_conv_layers-1):
            if ncl < self.n_stride2:
                stride  = 2
            else:
                stride  = 1
            self.conv_layers.append(conv_ln_lrelu(ch_in, ch_out, stride, pad=2))
            ch_in   *= 2
            ch_out  *= 2
            ncl     += 1

        self.conv_layers.append(
            nn.Conv2d(
                in_channels    = ch_in,
                out_channels   = 1,
                kernel_size    = 4,
            ),
        )

    def forward(self, x):
        for layer in self.conv_layers:
            x = layer(x)
        y = x.view(-1)
        return y

## Create a custom Dataset from the images in a folder

In [ ]:
class CustomDataSet(Dataset):

    def __init__(self, root_dir, transform):
        self.root_dir     = root_dir
        self.transform    = transform
        self.all_images   = os.listdir(root_dir)
        self.total_images = natsorted(self.all_images)

    def __len__(self):
        return len(self.total_images)

    def __getitem__(self, idx):
        img_loc      = os.path.join(self.root_dir, self.total_images[idx])
        image        = Image.open(img_loc).convert("RGB")
        tensor_image = self.transform(image)
        return tensor_image

### Define the transformation that will be applied to the images

- convert the images to tensors
- crop the images
- resize the images
- normalize the images. 

### Instantiate a Custom Dataset

### Create a training DataLoader

In [ ]:
def get_loader(crop_size, image_size, batch_size, dataset_train_dir):

    if crop_size != None:
        offset_height = (218 - crop_size) // 2
        offset_width  = (178 - crop_size) // 2
        crop = lambda x: x[:, offset_height:offset_height + crop_size, offset_width:offset_width + crop_size]

        train_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Lambda(crop),
                transforms.Resize(size=(config["image_size"], config["image_size"]), antialias=True),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ]
        )
    else:
        train_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Resize(size=(config["image_size"], config["image_size"]), antialias=True),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ]
        )


    train_data = CustomDataSet(
        root_dir  = train_dir,
        transform = train_transform,
    )

    train_loader = DataLoader(
        dataset     = train_data,
        batch_size  = config["batch_size"],
        shuffle     = True,
        drop_last   = True,
        num_workers = 4,
        pin_memory  = True,
    )
    return train_loader, train_data

## Let us check if everything works fine and display a few real images.

In [ ]:
def check_dataloader(crop_size, image_size, batch_size, dataset_train_dir):
    NR, NC    = 3, 3
    loader, _ = get_loader(crop_size, image_size, batch_size, dataset_train_dir)
    imgs      = next(iter(loader))
    print(f'Batch of images shape: {imgs.shape}')   # BS, Ch, H, W

    if NR*NC > imgs.shape[0]:
        NR = 2
        if NR*NC > imgs.shape[0]:
            NR = 1
            if NR*NC > imgs.shape[0]:
                NC = 2

    _, ax    = plt.subplots(NR, NC, figsize=(3*NC,3*NR))
    plt.suptitle(
        'Some real images of {config["dataset"]} dataset',
        fontsize=15,
        fontweight='bold'
    )

    index = 0
    for r in range(NR):
        for c in range(NC):
            index += 1
            if NR==1:
                ax[c].imshow((imgs[index].permute(1,2,0)+1)/2) 
            else:
                ax[r][c].imshow((imgs[index].permute(1,2,0)+1)/2) 

In [ ]:
check_dataloader(
    crop_size         = config["crop_size"],
    image_size        = config["image_size"], 
    batch_size        = config["batch_size"], 
    dataset_train_dir = train_dir,
)

## Functions to save and load the models to/from file

In [ ]:
def save_model_and_results(
        critic,
        generator,
        optimizer_c,
        optimizer_g,
        results,
        epoch,
        hyperparameters,
        file_name,
    ):
    results_to_save = {
        'critic':          critic.state_dict(),
        'generator':       generator.state_dict(),
        'c_optimizer':     optimizer_c.state_dict(),
        'g_optimizer':     optimizer_g.state_dict(),
        'results':         results,
        'epoch':           epoch,
        'hyperparameters': hyperparameters,
    }

    torch.save(
        results_to_save,
        file_name,
    )

In [ ]:
def load_model(critic, generator, optimizer_c, optimizer_g, file_name, device):
    '''
    Given instances of the generator and critic models, loads from file 'file_name':
    (i)   the weights of both models,
    (ii)  the optimizers state,
    (iii) the results obtained during model training and
    (iv)  the training hyperparameters used to train the models,
    and put the models on 'device'.

    Returns the loaded results and the loaded hyperparameters.
    '''

    results_loaded = torch.load(file_name)

    critic.load_state_dict(results_loaded['critic'])
    critic.to(device)

    generator.load_state_dict(results_loaded['generator'])
    generator.to(device)

    optimizer_c.load_state_dict(results_loaded['c_optimizer'])
    optimizer_g.load_state_dict(results_loaded['g_optimizer'])

    # Returns the saved results and the saved hyperparameters
    return results_loaded['results'], results_loaded['epoch'], results_loaded['hyperparameters']

## Instantiate the Models and select the loss function and optimizers

In [ ]:
critic    = Critic(
    config["c_hidden_dim"],
    config["image_size"],
    config["channels"],
    config["c_min_feature_size"],
    config["c_conv_layers"],
).to(device)

generator = Generator(
    config["z_dim"],
    config["g_hidden_dim"],
    config["image_size"],
    config["channels"],
    config["g_min_feature_size"],
    config["g_conv_layers"],
).to(device)

optimizer_c = torch.optim.Adam(
    critic.parameters(),
    lr    = config["lr_c"],
    betas = (config["beta1"], config["beta2"])
)
optimizer_g = torch.optim.Adam(
    generator.parameters(),
    lr    = config["lr_g"],
    betas = (config["beta1"], config["beta2"])
)

## Print the generator model summary

In [ ]:
from torchinfo import summary

aux_data = torch.randn(config['batch_size'], config["z_dim"], device=device)
summary(
    generator,
    input_data   = aux_data,
    col_width    = 16,
    col_names    = ["kernel_size", "output_size", "num_params"],
    row_settings = ["var_names"],
)

## Print the critic model summary

In [ ]:
from torchinfo import summary

aux_data = torch.randn(
    (
    config['batch_size'],
    config['channels'], 
    config['image_size'],
    config['image_size']
    )).to(device)

summary(
    critic,
    input_data   = aux_data,
    col_width    = 16,
    col_names    = ["kernel_size", "output_size", "num_params"],
    row_settings = ["var_names"],
)

## Train the WGAN-GP Model

In [ ]:
def gradient_penalty_v1(real_images, fake_images, critic, device):

    # interpolation
    shape = [real_images.size(0)] + [1] * (real_images.dim() - 1)
    alpha = torch.rand(shape).to(device)
    z = real_images + alpha * (fake_images.detach() - real_images)
    z.requires_grad = True

    # gradient penalty
    o = critic(z)
    grad_outputs = torch.ones(o.size()).to(device)
    g = grad(o, z, grad_outputs, create_graph=True)[0].view(z.size(0), -1)
    gp = ((g.norm(p=2, dim=1) - 1)**2).mean()

    return gp


def gradient_penalty_v2(real_images, fake_images, critic, device):

    batch_size = real_images.size(0)
    alpha      = torch.rand(batch_size, 1, 1, 1).to(device)

    interpolated = alpha * real_images + (1 - alpha) * fake_images.detach()
    interpolated.requires_grad = True

    critic_out = critic(interpolated)

    grad_values = torch.ones(critic_out.size()).to(device)
    gradients   = torch.autograd.grad(
        outputs      = critic_out,
        inputs       = interpolated,
        grad_outputs = grad_values,
        create_graph = True,
        retain_graph = True)[0]

    gradients = gradients.view(batch_size, -1)

    # calculate the norm of gradients, adding epsilon to prevent 0 values
    epsilon        = 1e-13
    gradients_norm = torch.sqrt(torch.sum(gradients ** 2, dim=1) + epsilon)

    gp = ((gradients_norm - 1) ** 2).mean()
    return gp

In [ ]:
def train_wgangp(
        num_epochs,
        critic,
        generator,
        optimizer_c,
        optimizer_g,
        latent_dim,
        train_loader,
        ncritic                 = 5,
        gradient_penalty        = True,
        gradient_penalty_weight = 10,
        log_interval            = 100,
        results                 = None,
        start_epoch             = 0,
        device                  = device,
    ):

    # Batch of latent (noise) vectors for evaluating
    # / visualizing the training progress of the generator

    ref_batch_size = 25
    mean_gp        = 0.0

    assert config["log_interval"] % ncritic == 0, \
        f'Log interval is {config["log_interval"]} and must be a multiple of {ncritic}'

    gen_log_interval = int(config["log_interval"] / ncritic)
    print(f'Log interval according to the generator iterations: {gen_log_interval}')

    if gradient_penalty is None:
        print('Using gradient clipping')
    else:
        if gradient_penalty=='v1':
            print('Using gradient penalty v1')
        elif gradient_penalty=='v2':
            print('Using gradient penalty v2')

    # Fixed input batch of random vectors to used to evaluate the generation
    # quality during training
    z_sample = torch.randn(ref_batch_size, config["z_dim"], device=device)

    for epoch in range(start_epoch, num_epochs):

        ts  = time.time()

        critic.train()
        generator.train()

        for i, imgs in enumerate(train_loader):

            step       = epoch * len(train_loader) + i + 1
            batch_size = imgs.size(0)

            # Real images
            real_images = imgs.to(device)

            # Generated (fake) images
            noise       = torch.randn(
                batch_size,
                latent_dim,
                device = device,
            )  # format BS,C,H,W
            fake_images = generator(noise)

            # --------------------------
            # Train the Critic
            # --------------------------

            optimizer_c.zero_grad()

            # Get the critic output on real images
            real_logits = critic(real_images).view(-1) # Nx1 -> N

            # Get the critic output on fake images
            fake_logits = critic(fake_images.detach()).view(-1)

            # Critic loss

            critic_loss = fake_logits.mean() - real_logits.mean()  # Wasserstein-1 Distance

            # -----------------------------------------
            # Gradient penalty

            if gradient_penalty=='v1':
                gp = gradient_penalty_v1(real_images, fake_images, critic, device)
                critic_loss  += gp * gradient_penalty_weight
            elif gradient_penalty=='v2':
                gp = gradient_penalty_v2(real_images, fake_images, critic, device)
                critic_loss  += gp * gradient_penalty_weight

            critic_loss.backward()

            optimizer_c.step()

            # Use weight clipping (standard Wasserstein GAN)
            if not gradient_penalty:
                for p in critic.parameters():
                    p.data.clamp_(-0.01, 0.01)

            if step % ncritic == 0:

                # --------------------------
                # Train the Generator
                # --------------------------

                optimizer_g.zero_grad()

                # Calculate Generator loss
                fake_logits = critic(fake_images).view(-1)
                gen_loss    = -fake_logits.mean()
                gen_loss.backward()

                optimizer_g.step()

                # ------------------------------------
                # Save the results in a dictionary
                # ------------------------------------

                results["c_loss"].append(critic_loss.item())
                results["g_loss"].append(gen_loss.item())
                if gradient_penalty:
                    results["gp"].append(gp.item())

            # Print progress metrics and save them to W&B ..........................
            if (i+1) % config["log_interval"] == 0:

                mean_c_loss      = np.mean(results["c_loss"][-gen_log_interval:])
                mean_g_loss      = np.mean(results["g_loss"][-gen_log_interval:])
                if gradient_penalty:
                    mean_gp          = np.mean(results["gp"][-config["log_interval"]:])

                print(f'epoch|iter: {epoch+1 :4d} | {i+1 :5d} / {len(train_loader) :6d}', end = "  ")
                print(f'({((i+1)*100)/len(train_loader) :0>5.1f}%)', end="  ")
                print(f'C loss: {mean_c_loss :0>9.6f}', end="  ")
                if gradient_penalty:
                    print(f'G loss: {mean_g_loss :0>9.6f}', end="  ")
                    print(f'GP: {mean_gp :0>.6f}')
                else:
                    print(f'G loss: {mean_g_loss :0>9.6f}')

                try:
                    # Log metrics to Weights & Biases ............................
                    wandb.log(
                        {
                        "c_loss":   mean_c_loss,
                        "g_loss":   mean_g_loss,
                        "gp":       mean_gp,
                        "epoch":    epoch+1,
                        }
                    )
                except Exception as ex:
                    print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

                # Save intermediate generated images ....................................
                generator.eval()
                f_imgs_sample = (generator(z_sample).data + 1) / 2.0

                save_dir = f'./results/{config["experiment_name"]}'
                save_image(
                    f_imgs_sample,
                    f'{save_dir}/{config["experiment_name"]}_epoch{str(epoch+1).zfill(3)}_iteration{str(i+1).zfill(5)}.png', 
                    nrow=5
                )

        te        = time.time()
        texec_sec = te - ts
        texec_str = time_format(texec_sec)
        print(f'Epoch training time: {texec_str}')
        results['epoch_training_time'].append(texec_sec)

        try:
            wandb.log(
                {
                "epoch_training_time_sec": texec_sec,
                }
            )
        except Exception as ex:
            print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

        if ((epoch+1) % config["checkp_interval"] == 0) or ((epoch+1) == config['epochs']):
            file_save_model = f'models/checkpoints/{config["experiment_name"]}_{str(epoch+1).zfill(3)}.pth'
            save_model_and_results(
                critic,
                generator,
                optimizer_c,
                optimizer_g,
                results,
                epoch,
                config,
                file_save_model,
            )

In [ ]:
# Create an empty dictionary to store the training results .................
results = {
    'c_loss':              [],
    'g_loss':              [],
    'gp':                  [],
    'epoch_training_time': [],
}

# Instantiate the training dataloader ......................................

train_loader, _ = get_loader(
    config["crop_size"],
    config["image_size"],
    config["batch_size"],
    train_dir,
)

In [ ]:
# =========================================================================
# Train the model from the beginning
# =========================================================================

if LOAD_TRAINED_MODEL == False and SKIP_TRAIN_MODEL == False:

    start_epoch = 0

    train_wgangp(
        config["epochs"],
        critic,
        generator,
        optimizer_c,
        optimizer_g,
        latent_dim              = config["z_dim"],
        train_loader            = train_loader,
        ncritic                 = config["ncritic"],
        gradient_penalty        = config["gradient_penalty"],
        gradient_penalty_weight = config["lambda_gp"],
        log_interval            = config["log_interval"],
        results                 = results,
        start_epoch             = start_epoch,
        device                  = device,
    )

# =========================================================================
# Load the saved models
# =========================================================================

elif LOAD_TRAINED_MODEL == True:

    file_save_model = f'models/{config["experiment_name"]}.pth'
    results, start_epoch, _ = load_model(
        critic,
        generator,
        optimizer_c,
        optimizer_g,
        file_save_model,
        device,
    )

    # ---------------------------------------------------------------------
    # Continue training of the loaded models
    # ---------------------------------------------------------------------

    if SKIP_TRAIN_MODEL == False:

        train_wgangp(
            config["epochs"],
            critic,
            generator,
            optimizer_c,
            optimizer_g,
            latent_dim              = config["z_dim"],
            train_loader            = train_loader,
            ncritic                 = config["ncritic"],
            gradient_penalty        = config["gradient_penalty"],
            gradient_penalty_weight = config["lambda_gp"],
            log_interval            = config["log_interval"],
            results                 = results,
            start_epoch             = start_epoch,
            device                  = device,
        )

## Export results to a CSV file

In [ ]:
import pandas as pd

results_df = pd.DataFrame(
    list(
        zip(
            results['c_loss'],
            results['g_loss'],
            results['gp']
        )
    ),
    columns = ['c_loss', 'g_loss', 'gp']
)
results_df.head()

file_name = f'results/{config["experiment_name"]}_results.csv'
results_df.to_csv(file_name)

## Generate grids of images with the fully trained generator

In [ ]:
def generate_grid_images(generator, num_grids, grid_W_H, config, device):

    grid_size = grid_W_H ** 2
    assert config["batch_size"] >= grid_size, f'Grid size must be less or equal to batch size={config["batch_size"]}'

    generator.eval()

    with torch.inference_mode():

        for num in range(num_grids):

            # Generate a set of latent vectors
            noise = torch.randn(config['batch_size'], config['z_dim'], device=device)

            # Generate a set of fake images with G
            fake = generator(noise).detach().cpu()

            if(config['batch_size'] > grid_size):
                fake = fake[:grid_size]

            # Create a grid with the generated images
            grid = make_grid(fake, padding=2, normalize=True)
            grid = grid.permute(1, 2, 0)
            grid = grid.numpy()

            # Display the grid of images
            _ = plt.figure(figsize=(10, 10), constrained_layout=True)
            plt.imshow(grid)

            # Save the grid of images as a PNG file
            file_png = f'results/{config["experiment_name"]}/{config["experiment_name"]}_generated_final_{str(num+1).zfill(3)}.png'
            plt.imsave(file_png, grid)

In [ ]:
generate_grid_images(generator, 8, 8, config, device)

In [ ]:
# Mark the Weights & Bias run as finished
wandb.finish()